In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version/val/nofire/ashton-morris-cEn0ztKTjes-unsplash.jpg
/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version/val/nofire/elena-mozhvilo-tvDuKySFy_o-unsplash.jpg
/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version/val/nofire/kristaps-ungurs-TlJSzJlPRc8-unsplash.jpg
/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version/val/nofire/nitish-meena-ytGtKR94nAI-unsplash.jpg
/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version/val/nofire/guillaume-bourdages-VKMx4lonuLg-unsplash.jpg
/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version/val/nofire/nathan-anderson-0mZLht43A_c-unsplash.jpg
/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version/val/nofire/chad-madden-9HDfRHhCxME-unsplash.jpg
/kaggle/input/datasets/elmadafri/the-wil

In [2]:
import os, re
import numpy as np
import pandas as pd
from PIL import Image

BASE = '/kaggle/input/datasets/elmadafri/the-wildfire-dataset/the_wildfire_dataset_2n_version'
Image.MAX_IMAGE_PIXELS = 200_000_000     # trusted source; silences bomb warnings

VALID_EXT = {'.jpg', '.jpeg', '.png'}

def source_and_group(name):
    stem = re.sub(r' \(\d+\)(?=\.\w+$)', '', name)        # strip " (1)" download suffix
    if re.search(r'-unsplash\.\w+$', stem):
        return 'unsplash', '-'.join(stem.split('-')[:2])
    head = stem.split('_')[0]
    if head.isdigit():
        return 'flickr', head
    return 'other', stem

rows, bad, skipped = [], [], []
for split in ['train', 'val', 'test']:
    for cls in ['fire', 'nofire']:
        d = f'{BASE}/{split}/{cls}'
        for n in sorted(os.listdir(d)):
            if os.path.splitext(n)[1].lower() not in VALID_EXT:
                skipped.append(f'{split}/{cls}/{n}')
                continue
            src, key = source_and_group(n)
            p = f'{d}/{n}'
            try:
                with Image.open(p) as im:
                    w, h, fmt, mode = im.size[0], im.size[1], im.format, im.mode
            except Exception as e:
                bad.append((p, repr(e)))
                continue
            rows.append(dict(orig_split=split, cls=cls, src=src, rawkey=key,
                             name=n, path=p, w=w, h=h, fmt=fmt, mode=mode))

df = pd.DataFrame(rows)
df['short'] = df[['w', 'h']].min(axis=1)

print(f'{len(df)} images | {len(skipped)} non-image files skipped | {len(bad)} unreadable')
for s in skipped:
    print('  skipped:', s)

print('\nsource x class:')
print(pd.crosstab(df.src, df.cls).to_string())

print('\nformat x (source, class):')
print(pd.crosstab(df.fmt, [df.src, df.cls]).to_string())

print('\nimages smaller than 256 on the short side:')
print(df[df.short < 256][['orig_split','cls','src','w','h','name']].to_string(index=False))

df.to_csv('/kaggle/working/inventory.csv', index=False)
print('\nsaved -> /kaggle/working/inventory.csv')

2699 images | 1 non-image files skipped | 0 unreadable
  skipped: val/fire/desktop.ini

source x class:
cls       fire  nofire
src                   
flickr     989     198
unsplash    56    1456

format x (source, class):
src  flickr        unsplash       
cls    fire nofire     fire nofire
fmt                               
JPEG    933    195       56   1456
MPO      40      1        0      0
PNG      16      2        0      0

images smaller than 256 on the short side:
orig_split  cls    src   w   h                         name
     train fire flickr 300 225 37342480522_2dfdd4f170_o.jpg
     train fire flickr 153 206 51341756029_4e29bba961_o.jpg
      test fire flickr 180 240 52290796314_c886e9fd0e_o.jpg
      test fire flickr 620 229 52293015808_1e4f9d5b0a_o.jpg

saved -> /kaggle/working/inventory.csv


In [3]:
import random
from collections import defaultdict

WINDOW, SEED = 100_000, 42

# group key: flickr -> upload session, unsplash -> photographer, other -> itself
recs = df.to_dict('records')
fl = sorted([r for r in recs if r['src'] == 'flickr'], key=lambda r: int(r['rawkey']))
sess, prev = 0, None
for r in fl:
    pid = int(r['rawkey'])
    if prev is not None and pid - prev > WINDOW:
        sess += 1
    r['group'], prev = f'flickr_s{sess}', pid
for r in recs:
    if r['src'] == 'unsplash':
        r['group'] = f"unsplash_{r['rawkey']}"
    elif r['src'] == 'other':
        r['group'] = f"other_{r['name']}"

groups = defaultdict(list)
for r in recs:
    groups[r['group']].append(r)
print(f'{len(groups)} groups over {len(recs)} images')

def stratum(members):
    fire = sum(1 for m in members if m['cls'] == 'fire')
    return (members[0]['src'], 'fire' if fire * 2 >= len(members) else 'nofire')

by_stratum = defaultdict(list)
for g, m in groups.items():
    by_stratum[stratum(m)].append((g, m))

# ~15/15 split, EXCEPT flickr/nofire which is deliberately over-allocated to
# test+val so a source-matched evaluation slice exists.
TARGETS = {
    ('flickr',   'nofire'): {'test':  90, 'val': 30},
    ('flickr',   'fire'  ): {'test': 149, 'val': 148},
    ('unsplash', 'nofire'): {'test': 219, 'val': 218},
    ('unsplash', 'fire'  ): {'test':   9, 'val':   8},
}

assign, rng = {}, random.Random(SEED)
for strat, glist in by_stratum.items():
    rng.shuffle(glist)
    need = dict(TARGETS.get(strat, {'test': 0, 'val': 0}))
    for g, m in glist:
        for s in ['test', 'val']:
            if need.get(s, 0) > 0:
                assign[g] = s
                need[s] -= len(m)
                break
        else:
            assign[g] = 'train'

for r in recs:
    r['split'] = assign[r['group']]

sp = pd.DataFrame(recs)[['name', 'cls', 'src', 'group', 'split', 'orig_split', 'w', 'h']]

# verification
assert sp.groupby('group').split.nunique().max() == 1, 'a group spans splits'
print('\nno group spans a split ✓\n')
print(pd.crosstab([sp.src, sp.cls], sp.split).to_string())
print('\nsplit sizes:\n', sp.split.value_counts().to_string())
print('\nsource-matched TEST slice (flickr only):')
print(sp[(sp.split == 'test') & (sp.src == 'flickr')].cls.value_counts().to_string())

sp.to_csv('/kaggle/working/split_v1.csv', index=False)
print('\nsaved -> /kaggle/working/split_v1.csv   (download and commit to the repo)')

1445 groups over 2699 images

no group spans a split ✓

split            test  train  val
src      cls                     
flickr   fire     152    688  149
         nofire    95     73   30
unsplash fire       7     28   21
         nofire   222   1014  220

split sizes:
 split
train    1803
test      476
val       420

source-matched TEST slice (flickr only):
cls
fire      152
nofire     95

saved -> /kaggle/working/split_v1.csv   (download and commit to the repo)


In [4]:
import time

OUT, SHORT, Q = '/kaggle/working/resized', 256, 92
os.makedirs(OUT, exist_ok=True)

def out_name(name):
    return os.path.splitext(name)[0] + '.jpg'

# make sure no two source files collide on the same output name
outs = [out_name(r['name']) for r in recs]
assert len(set(outs)) == len(outs), 'output filename collision'

def resize_one(src, dst, short=SHORT, quality=Q):
    with Image.open(src) as im:
        im.draft('RGB', (short * 2, short * 2))
        im = im.convert('RGB')
        w, h = im.size
        s = short / min(w, h)
        im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
        im.save(dst, 'JPEG', quality=quality)

t0 = time.time()
for i, r in enumerate(recs):
    dst = f"{OUT}/{out_name(r['name'])}"
    if not os.path.exists(dst):
        resize_one(r['path'], dst)
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(recs)}  {time.time()-t0:.0f}s")

files = os.listdir(OUT)
mb = sum(os.path.getsize(f'{OUT}/{f}') for f in files) / 1e6
print(f'\n{len(files)} files, {mb:.0f} MB, {time.time()-t0:.0f}s total')

# record the output filename in the split, and re-save
sp['out_name'] = sp['name'].map(out_name)
sp.to_csv('/kaggle/working/split_v1.csv', index=False)
print('split_v1.csv updated with out_name column')

# verify every row in the split has a matching file on disk
missing = [n for n in sp.out_name if not os.path.exists(f'{OUT}/{n}')]
print(f'missing files: {len(missing)}')

# spot-check a few outputs
for n in list(sp.out_name)[:3]:
    with Image.open(f'{OUT}/{n}') as im:
        print(f'  {n}  {im.size}  {im.mode}  {im.format}')

  500/2699  61s
  1000/2699  142s
  1500/2699  271s
  2000/2699  376s
  2500/2699  473s

2699 files, 82 MB, 530s total
split_v1.csv updated with out_name column
missing files: 0
  11713547914_dd11630b77_o.jpg  (341, 256)  RGB  JPEG
  11826515394_7959916eff_o.jpg  (341, 256)  RGB  JPEG
  11875858055_b291d73a36_o.jpg  (341, 256)  RGB  JPEG


In [5]:
from IPython.display import FileLink
FileLink('split_v1.csv')

/kaggle/working/split_v1.csv